# Tiny CPU model workflow

Builds a **reduced-filter** 3D CNN (far smaller than the paper's `(128, 128, 50)` input / `(64, 64, 128, 256)` filters) and runs one bounded train -> evaluate -> save/reload flow on tiny synthetic arrays, CPU-only, in a few seconds. This is **not** a reproduction of the paper's results: it is a smoke-sized tour of the `multimodal_ad.models` API.

Requires the `model` extra: run `just install` (installs it along with everything else), or `uv sync --extra model` for a lighter install. All logic lives in `multimodal_ad.models` (see [`src/multimodal_ad/models/__init__.py`](../src/multimodal_ad/models/__init__.py)); this notebook only calls it and displays results.

In [ ]:
from pathlib import Path
import tempfile

import matplotlib.pyplot as plt
import numpy as np

from multimodal_ad.models.architecture import Cnn3DConfig, build_3d_cnn
from multimodal_ad.models.evaluation import evaluate_predictions
from multimodal_ad.models.training import TrainingConfig, load_model, save_model, train_model

## Tiny synthetic training arrays

Random noise, not real scans; only shapes/dtypes matter here.

In [ ]:
rng = np.random.default_rng(1234)
SIZE = 48  # far smaller than the paper's 128x128x50 input, CPU-friendly

x_train = rng.random((8, SIZE, SIZE, SIZE, 1)).astype("float32")
y_train = (np.arange(8) % 2).astype("float32")
x_val = rng.random((4, SIZE, SIZE, SIZE, 1)).astype("float32")
y_val = (np.arange(4) % 2).astype("float32")

## Build a reduced-filter 3D CNN

Same four-block topology as the paper's architecture, tiny filter counts.

In [ ]:
config = Cnn3DConfig(
    width=SIZE,
    height=SIZE,
    depth=SIZE,
    filters=(4, 4, 8, 8),  # paper default: (64, 64, 128, 256)
    dense_units=16,  # paper default: 512
    name="tiny-cnn",
)
model = build_3d_cnn(config)
model.summary()

## Train for one bounded epoch

`epochs=1` and no early-stopping baseline: this is a workflow demo, not a real training run.

In [ ]:
output_dir = Path(tempfile.mkdtemp(prefix="multimodal-ad-tiny-model-"))
training_config = TrainingConfig(
    epochs=1,
    batch_size=2,
    checkpoint_path=output_dir / "tiny.keras",
    early_stopping_baseline=None,
    verbose=0,
)
result = train_model(model, x_train, y_train, x_val, y_val, training_config)
result.val_loss, result.val_accuracy

## Visualize training history

`epochs=1` means one point per metric; the plot still shows the loss/accuracy
values the single bounded epoch produced (a real run would show curves over
many epochs).


In [ ]:
history = result.history.history

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for name in ("loss", "val_loss"):
    if name in history:
        axes[0].plot(history[name], marker="o", label=name)
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

for name in ("accuracy", "val_accuracy"):
    if name in history:
        axes[1].plot(history[name], marker="o", label=name)
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()

fig.tight_layout()
plt.show()


## Correct evaluation metrics

Accuracy/sensitivity/specificity/AUC from thresholded probabilities (`multimodal_ad.models.evaluation`).

In [ ]:
probabilities = result.model.predict(x_val, verbose=0).reshape(-1)
metrics = evaluate_predictions(y_val, probabilities)
metrics

## Predicted probability vs. label

Predicted probability (thresholded at 0.5, per `multimodal_ad.models.evaluation`)
against the ground-truth label for each of the 4 validation samples; useful
as a quick visual sanity check, not a substitute for the metrics above.


In [ ]:
sample_indices = np.arange(len(y_val))
plt.figure(figsize=(6, 3))
plt.bar(sample_indices - 0.15, probabilities, width=0.3, label="predicted probability")
plt.bar(sample_indices + 0.15, y_val, width=0.3, label="true label")
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="0.5 threshold")
plt.xticks(sample_indices)
plt.xlabel("validation sample")
plt.legend()
plt.title("Predicted probability vs. label (tiny, one-epoch model)")
plt.show()


## Save, reload, and confirm identical predictions

In [ ]:
final_path = output_dir / "tiny-final.keras"
save_model(result.model, final_path)
reloaded_model = load_model(final_path)

bool(np.allclose(
    result.model.predict(x_val, verbose=0),
    reloaded_model.predict(x_val, verbose=0),
))

## Next steps

- See `03-explainability.ipynb` for Grad-CAM and AAL2 region ranking on top of a model like this one.
- `docs/reproducibility.md` documents exactly what is and isn't validated against the paper's published numbers.